# Testing YOLO on Single Image from Simulator
### Starting image
![simulator gate](start.png)

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLO26 nano model
model = YOLO("yolo26n.pt")

# Run inference on an image
results = model("start.png")

# Display results
results[0].show()


image 1/1 c:\Users\annie\PersonalProjects\dabjak4\ai-grand-prix\start.png: 384x640 1 train, 118.7ms
Speed: 5.7ms preprocess, 118.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)


Results with no training are shown below:

![no training yolo26 detection](yolo-detection-no-training.png)

Train: predicted class

0.25: confidence score

# Training with Online Dataset

## Download the Dataset

Note: the following code just downloads the dataset, so once you have it, you can just comment it out if you don't want it to run again. If it does run again, it won't do anything. If the install was incomplete, delete the folder and rerun the code.

In [ ]:
# No need to run this if you already have AirSim-Drone-Racing-Lab-Gates folder

import os
from roboflow import Roboflow
from dotenv import load_dotenv

load_dotenv()

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
rf = Roboflow(ROBOFLOW_API_KEY)
project = rf.workspace("drone-racing").project("airsim-drone-racing-lab-gates")
version = project.version(2)
dataset = version.download("yolo26")

Dataset includes training set, validation set, and test set. Use the training set for learning, the validation set to check on training, and the test set at the end of training.

The labels are basically the boxes that were manually traced around the images from the dataset (annotations included as part of it). They each have info in the form "0 0.421875 0.496875 0.41640625 0.84453125". The first number is the corresponding image number. The next 2 numbers are the coordinates of the top left corner and the last 2 numbers are the coordinates of the bottom right corner.

Also make sure that the paths in data.yaml for test, train, and valid are correct.

## Training

Note: there's a lot of parameters for model training (see docs in Resources), so I only used what I expect to make the largest impact. If it needs refining, we could look at others. 

In [ ]:
# pretrained model from YOLO
model = YOLO("yolo26n.pt")

In [ ]:
# train
trained_model = model.train(
    data="datasets/synthetic-gates/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=5,
    seed=0,
    cache=True,
    name="n4_train"
)

In [ ]:
# choose the model to use
trained_model = YOLO("models/n5.pt")

# run inference on image. conf level means it will only show boxes with a confidence level above the chosen level
start_results = trained_model("start.png", conf=0.5)

# print number of boxes detected by the model
print("Test data results:", len(start_results[0].boxes), "detections")

rotated_results = trained_model("start_rotated_enlarged.png", conf=0.08)
print("Sim results:", len(rotated_results[0].boxes), "detections")

# display bounded boxes on images as pngs
start_results[0].show()
rotated_results[0].show()

* patience (default 100): number of epochs with no validation improvement before early stopping
* batch (default 16): batch size
* epochs (default 100): number of epochs
* imgsz (default 640 px): image size
* seed (default 0): sets the seed

In [ ]:
# combining datasets
# using AirSim-Drone-Racing-Lab-Gates-2 and synthetic-gates

# choose the pretrained model you want to start with (can be any)
base_model = YOLO("models/n5.pt")

# train
merged_model = base_model.train(
    data="datasets/merged2/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=5,
    workers=12,
    seed=0,
    cache=True,
    name="n6_train"
)

Before running the next code block, move a copy of runs/nx_train/weights/best.pt into models and name it it's corresponding model id (nx.pt). If there should only be one model, but there is 2, choose the last on (e.g. n2_train and n2_train-2, choose n2_train_2).

In [ ]:
# choose the model to use
result_model = YOLO("models/n6.pt")

# run the model on a chosen image
start_results = result_model("start.png", conf=0.5)

# prints the number of boxes detected
print("Test data results:", len(start_results[0].boxes), "detections")

rotated_results = result_model("start__rotated_enlarged.png", conf=0.05)
print("Sim results:", len(rotated_results[0].boxes), "detections")

# display boxed results as pngs
start_results[0].show()
rotated_results[0].show()


image 1/1 c:\Users\annie\PersonalProjects\dabjak4\ai-grand-prix\start.png: 384x640 4 gates, 45.6ms
Speed: 1.2ms preprocess, 45.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
Test data results: 4 detections

image 1/1 c:\Users\annie\PersonalProjects\dabjak4\ai-grand-prix\start_enlarged.png: 384x640 1 gate, 37.9ms
Speed: 1.4ms preprocess, 37.9ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)
Sim results: 1 detections


merged1: combination of AirSim-Drone-Racing-Lab-Gates-2 and synthetic-gates. synthetic-gates are weighted heavier, so dataset is tripled in the merged dataset (previously about half of the other dataset).

### Considerations for Model Improvement

1. Take N5 and adjust the weighting for each dataset.
2. We can try changing the imgsz value (see issue #2 in the repo). I think this could potentially matter a lot for VQ2.
3. We can change the learning rate to see if that makes a difference, if improvements to the model are needed.
4. We can technically choose to not use the pretrained model, although this likely won't result in any improvements.

### Considerations for Faster Model Learning

1. Using the workers parameter sets the number of threads to run at once. The default is 8, so if your GPU can handle it, try setting it higher.
2. Setting cache to True to allow caching. The default is False.

## Training Results

| Base Model | Model Name | Parameters | Confidence Level for Closest Gate in start.png | Time to Process start.png | Training Time | Notes |
| - | - | - | - | - | - | - |
| Normal YOLO26 | N1 | data="AirSim-Drone-Racing-Lab-Gates-2/data.yaml", epochs=100, imgsz=640, batch=16, patience=100, seed=0 | N/A | 1.6ms preprocess, 43.6ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640) | 2+ hours | Doesn't detect gates because the training dataset envrionment is too different from the simulator environment. |
| Normal YOLO26 | N2 | data="AirSim-Drone-Racing-Lab-Gates-2/data.yaml", epochs=100, imgsz=640, batch=16, patience=5, seed=0, cache=True, hsv_h=0.5, hsv_s=0.9, hsv_v=0.6, name="n2_train" | N/A | Speed: 1.4ms preprocess, 43.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640) | 41m 42.7s | Doesn't detect gates. Was supposed to randomize coloring to make it less important |
| Normal YOLO26 | N3 | data="gen_dataset/dataset.yaml", epochs=100, imgsz=640, batch=16, patience=5, seed=0, cache=True, sname="n3_train" | 0.08 | Speed: 1.2ms preprocess, 32.3ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640) | 7m 25.3s | Very low confidence score. Used a dataset produced by transformation images of start.png (using albumentations).|
| Normal YOLO26 | N4 | data="synthetic-gates/data.yaml", epochs=100, imgsz=640, batch=16, patience=5, seed=0, cache=True, sname="n4_train" | 0.98 | Speed: 1.2ms preprocess, 32.3ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640) | 29m 11.9s | Uses data generated by Claude. Model can detect gates well if they are fully "upright." Has trouble with rotations. |
| N1 | N5 | data="merged1/data.yaml", epochs=100, imgsz=640, batch=16, patience=5, seed=0, cache=True, sname="n5_train" | 0.20 | Speed: 1.2ms preprocess, 45.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640) | 257m 40.1s | Trained model N1 on a combined dataset from N1 and N4. Has some trouble with rotated gates, but detects them at a lower confidence interval. |
| N5 | N6 |  |  |  |  | Training using Claude generated data and augmented data using albumentations. |

Notes: using color randomization (e.g. changing hsv_h values) seems to make training faster, but I'm not sure it if affects the accuracy of the model.

# Resources

## Yolo v26 (and Related)

YOLO v26 usage docs: https://docs.ultralytics.com/models/yolo26#usage-examples

YOLO v26 training docs (e.g. train parameters): https://docs.ultralytics.com/modes/train#musgd-optimizer

Roboflow training dataset: https://universe.roboflow.com/drone-racing/airsim-drone-racing-lab-gates/dataset/2

## About Machine Learning

Epochs and batches: https://www.geeksforgeeks.org/machine-learning/epoch-in-machine-learning/